# 98 — GNN Train: Run 12 Config

**Best run from autoresearch sweep (F1=0.31 on 22-positive test set)**

| Param | Value |
|---|---|
| hidden_dim | 128 |
| Loss | Focal (γ=2, α=0.9) |
| Epochs | 50 |
| Train samples | 1000 |
| Dropout | 0.0 |
| Action features | 3-dim: [log(min_ships), log(max_ships), eta/9] |

In [1]:
%run 97-library.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from sklearn.metrics import accuracy_score, classification_report
import math, os

In [2]:
print("Generating train dataset (1000 samples)...")
train_dataset = [generate_sample_97(i) for i in range(1000)]
print("Generating test dataset (100 samples)...")
test_dataset  = [generate_sample_97(10000 + i) for i in range(100)]

train_pos = sum(int(d['action'].y.sum()) for d, _, _ in train_dataset)
train_tot = sum(d['action'].x.shape[0] for d, _, _ in train_dataset)
test_pos  = sum(int(d['action'].y.sum()) for d, _, _ in test_dataset)
test_tot  = sum(d['action'].x.shape[0] for d, _, _ in test_dataset)
print(f"Train: {train_tot} action nodes, {train_pos} positive ({100*train_pos/max(train_tot,1):.1f}%)")
print(f"Test:  {test_tot} action nodes, {test_pos} positive ({100*test_pos/max(test_tot,1):.1f}%)")

# Verify 3-dim action features
assert train_dataset[0][0]['action'].x.shape[1] == 3, "Expected 3-dim action features"
print(f"Action feature dim: {train_dataset[0][0]['action'].x.shape[1]}  (min_ships, max_ships, eta/9)")

Generating train dataset (1000 samples)...


Currently using testing _04_score_and_decide
From 2, To 7 at step 4 with 45 ships (target has min 71)
From 1, To 10 at step 7 with 48 ships (target has min 67)
From 0, To 6 at step 5 with 58 ships (target has min 69)


<string>:656: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


Currently using testing _04_score_and_decide
From 4, To 6 at step 9 with 21 ships (target has min 44)
From 1, To 13 at step 2 with 57 ships (target has min 68)
From 0, To 13 at step 6 with 57 ships (target has min 89)


Currently using testing _04_score_and_decide
From 0, To 2 at step 4 with 82 ships (target has min 84)
From 0, To 4 at step 7 with 56 ships (target has min 84)


...

Currently using testing _04_score_and_decide
From 0, To 4 at step 4 with 43 ships (target has min 66)


<string>:656: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


Currently using testing _04_score_and_decide
From 3, To 6 at step 4 with 52 ships (target has min 94)
From 1, To 6 at step 2 with 51 ships (target has min 89)
From 1, To 6 at step 3 with 16 ships (target has min 89)


Currently using testing _04_score_and_decide
From 0, To 5 at step 5 with 21 ships (target has min 41)


Currently using testing _04_score_and_decide
From 1, To 3 at step 6 with 19 ships (target has min 72)
Train: 39503 action nodes, 378 positive (1.0%)
Test:  3328 action nodes, 22 positive (0.7%)
Action feature dim: 3  (min_ships, max_ships, eta/9)


In [3]:
HIDDEN_DIM = 128
DROPOUT   = 0.0
MODEL_PATH = "98-model.pt"

class GNNActionSelector(nn.Module):
    """Tripartite GNN: 3 SAGEConv passes, 2 output heads per action node.
    Action features (3-dim): [log(min_ships)/log(1024), log(max_ships)/log(1024), eta/9]
    """
    def __init__(self, hidden_dim=64, dropout=0.0):
        super().__init__()
        H = hidden_dim
        self.dropout = nn.Dropout(dropout)
        self.master_lin = nn.Linear(6,  H)
        self.planet_lin = nn.Linear(22, H)
        self.action_lin = nn.Linear(3,  H)

        self.sage1_pa = SAGEConv(H, H)
        self.sage1_pm = SAGEConv(H, H)
        self.sage1_mp = SAGEConv(H, H)
        self.sage2_ap = SAGEConv(H, H)
        self.sage3_pa = SAGEConv(H, H)

        self.select_head = nn.Linear(H, 1)
        self.ships_head  = nn.Sequential(nn.Linear(H, 1), nn.Sigmoid())

    def forward(self, data):
        h_m = self.dropout(torch.relu(self.master_lin(data['master'].x)))
        h_p = self.dropout(torch.relu(self.planet_lin(data['planet'].x)))
        h_a = self.dropout(torch.relu(self.action_lin(data['action'].x)))

        ei_spawns  = data['planet', 'spawns',    'action'].edge_index
        ei_attacks = data['action', 'attacks',   'planet'].edge_index
        ei_pm      = data['planet', 'to_master', 'master'].edge_index
        ei_mp      = data['master', 'to_planet', 'planet'].edge_index

        h_a = self.dropout(torch.relu(self.sage1_pa((h_p, h_a), ei_spawns)))
        h_m = self.dropout(torch.relu(self.sage1_pm((h_p, h_m), ei_pm)))
        h_p = self.dropout(torch.relu(self.sage1_mp((h_m, h_p), ei_mp)))
        h_p = self.dropout(torch.relu(self.sage2_ap((h_a, h_p), ei_attacks)))
        h_a = self.dropout(torch.relu(self.sage3_pa((h_p, h_a), ei_spawns)))

        return self.select_head(h_a).squeeze(-1), self.ships_head(h_a).squeeze(-1)


def focal_loss(logits, targets, gamma=2.0, alpha=0.9):
    """Focal loss. alpha>0.5 favours minority (attack) class."""
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
    p_t = torch.exp(-bce)
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
    return (alpha_t * (1 - p_t) ** gamma * bce).mean()


model = GNNActionSelector(hidden_dim=HIDDEN_DIM, dropout=DROPOUT)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model: {n_params:,} params  hidden_dim={HIDDEN_DIM}  dropout={DROPOUT}")

data0, _, _ = train_dataset[0]
if data0['action'].x.shape[0] > 0:
    sl, sh = model(data0)
    assert sl.shape == (data0['action'].x.shape[0],)
    print(f"Forward pass OK — {data0['action'].x.shape[0]} action nodes")

if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, weights_only=True))
    model.eval()
    _model_loaded = True
    print(f"Loaded model from {MODEL_PATH}")
else:
    _model_loaded = False
    print("No saved model — will train from scratch.")

Model: 169,090 params  hidden_dim=128  dropout=0.0
Forward pass OK — 31 action nodes
No saved model — will train from scratch.


In [4]:
GAMMA    = 2.0
ALPHA    = 0.9
N_EPOCHS = 50
LR       = 1e-3

if _model_loaded:
    print("Model already loaded — skipping training.")
else:
    print(f"Training: focal γ={GAMMA} α={ALPHA}  lr={LR}  epochs={N_EPOCHS}")
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    mse = nn.MSELoss()

    model.train()
    for epoch in range(N_EPOCHS):
        total_loss = total_sel = total_shp = 0.0
        n_graphs = 0
        for data, _, _ in train_dataset:
            if data['action'].x.shape[0] == 0:
                continue
            optimizer.zero_grad()
            sel_logit, ships_pred = model(data)
            loss_sel = focal_loss(sel_logit, data['action'].y, gamma=GAMMA, alpha=ALPHA)
            loss_shp = mse(ships_pred, data['action'].ships_target)
            loss = loss_sel + loss_shp
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            total_sel  += loss_sel.item()
            total_shp  += loss_shp.item()
            n_graphs   += 1
        if epoch % 10 == 0 or epoch == N_EPOCHS - 1:
            print(f"Epoch {epoch:3d} | total {total_loss/n_graphs:.4f} | focal {total_sel/n_graphs:.4f} | mse {total_shp/n_graphs:.4f}")

    torch.save(model.state_dict(), MODEL_PATH)
    print(f"Saved to {MODEL_PATH}")

Training: focal γ=2.0 α=0.9  lr=0.001  epochs=50


Epoch   0 | total 0.0058 | focal 0.0033 | mse 0.0025


Epoch  10 | total 0.0027 | focal 0.0024 | mse 0.0003


Epoch  20 | total 0.0025 | focal 0.0022 | mse 0.0003


Epoch  30 | total 0.0023 | focal 0.0020 | mse 0.0003


Epoch  40 | total 0.0022 | focal 0.0019 | mse 0.0003


Epoch  49 | total 0.0022 | focal 0.0020 | mse 0.0003
Saved to 98-model.pt


In [5]:
model.eval()
all_preds, all_labels = [], []
ships_abs_errors = []

with torch.no_grad():
    for data, _, _ in test_dataset:
        if data['action'].x.shape[0] == 0:
            continue
        sel_logit, ships_pred = model(data)
        preds  = (torch.sigmoid(sel_logit) > 0.5).int().tolist()
        labels = data['action'].y.int().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels)
        pos_mask = data['action'].y.bool()
        if pos_mask.any():
            ships_abs_errors.extend(
                (ships_pred[pos_mask] - data['action'].ships_target[pos_mask]).abs().tolist()
            )

print(f"Test: {sum(all_labels)} positives / {len(all_labels)} total action nodes\n")
print(f"Select Accuracy: {accuracy_score(all_labels, all_preds):.3f}\n")
print(classification_report(all_labels, all_preds, target_names=['skip', 'attack']))
if ships_abs_errors:
    print(f"Ships MAE (normalised): {sum(ships_abs_errors)/len(ships_abs_errors):.4f}")

Test: 22 positives / 3328 total action nodes

Select Accuracy: 0.974

              precision    recall  f1-score   support

        skip       1.00      0.97      0.99      3306
      attack       0.18      0.82      0.29        22

    accuracy                           0.97      3328
   macro avg       0.59      0.90      0.64      3328
weighted avg       0.99      0.97      0.98      3328

Ships MAE (normalised): 0.0840
